# 銘柄IDマスタ（01_IDmap）生成コード 概要

## 目的
東証上場銘柄一覧CSVを入力として、IRBANKの企業ID（EID）・URL・社名を取得し、
機械的な銘柄マスタ（01_IDmap.csv）を生成・更新する。

## インプット
- Data/Input/data_j_251218.csv  
  - 使用列：コード / 市場・商品区分 / 33業種区分

## アウトプット
- Data/01_IDmap.csv  
  - 列：code, ID, URL, name, UPDATE, category, status, industry_33

## 主な機能
- 市場・商品区分から category（株 / ETF / REIT / PRO）を自動判定
- 33業種区分（industry_33）を入力CSVから同期保存
- 株のみ IRBANK から EID・URL・社名を取得
- 取得済み（status=ok）の銘柄は再取得しない
- 処理中断に強い再実行安全設計（pending / not_found は再挑戦）

## status の意味
- ok：取得完了
- pending：取得処理中（中断耐性用の一時状態）
- not_found：株だが EID 未取得
- skip：ETF / PRO など取得対象外

## 備考
- 選定・優待・配当などの人手ラベルは扱わない（別コードで管理予定）

In [2]:
# (1) インポート_標準/外部ライブラリ一括
import os
import re
import csv
import time
import random
import tempfile
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Dict, List, Optional, Tuple

import requests
from requests.adapters import HTTPAdapter, Retry
from bs4 import BeautifulSoup
import pandas as pd


# (2) 設定_URL/UA/待機/タイムゾーン/入出力
BASE_BY_CODE = "https://irbank.net/{code}"
SEARCH_URL   = "https://irbank.net/search?q={code}"
RESULTS_URL  = "https://irbank.net/{eid}/results"

UA = "Mozilla/5.0 (compatible; eid-fetcher/0.3)"
SLEEP_BASE = 1.2
TZ = ZoneInfo("Asia/Tokyo")

LISTING_CSV = os.path.join("Data", "Input", "data_j_251218.csv")
IDMAP_CSV   = os.path.join("Data", "01_IDmap.csv")  # ★出力：category/status 追加版


# (3) HTTP/スクレイプ系ユーティリティ_関数群
# (3-1) セッション生成_Retry/Pool
def make_session(user_agent: str = UA) -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": user_agent})
    retries = Retry(
        total=5,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries, pool_maxsize=10)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

# (3-2) 丁寧な待機_ジッター付き
def polite_sleep(base: float = SLEEP_BASE) -> None:
    time.sleep(max(0.0, base + random.uniform(-0.6 * base, 0.6 * base)))

# (3-3) HTML取得_エンコ調整
def fetch_html(url: str, sess: requests.Session) -> str:
    r = sess.get(url, timeout=20, allow_redirects=True)
    r.raise_for_status()
    r.encoding = r.apparent_encoding or "utf-8"
    return r.text

# (3-4) 銘柄名抽出_h1からコード除去
def extract_name_from_html(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "html.parser")
    h1 = soup.find("h1")
    if not h1:
        return None
    text = h1.get_text(strip=True)
    text = re.sub(r"^[0-9A-Za-z]+\s*", "", text)  # 先頭のコード（英数字）除去
    return text or None

# (3-5) EID解決_コードからEIDへ（直URL→検索）
def resolve_eid_from_code(code: str, sess: requests.Session, wait: float = SLEEP_BASE) -> Optional[str]:
    # 直URL
    try:
        html = fetch_html(BASE_BY_CODE.format(code=code), sess)
        soup = BeautifulSoup(html, "html.parser")
        a = soup.select_one('a[href^="/E"][href*="/results"]') or soup.select_one('a[href^="/E"]')
        if a and a.get("href"):
            m = re.search(r"/(E\d+)", a["href"])
            if m:
                return m.group(1)
    except Exception:
        pass

    # 検索
    try:
        polite_sleep(wait)
        html2 = fetch_html(SEARCH_URL.format(code=code), sess)
        soup2 = BeautifulSoup(html2, "html.parser")
        for a in soup2.select('a[href^="/E"]'):
            m = re.search(r"^/?(E\d+)", a.get("href", ""))
            if m:
                return m.group(1)
    except Exception:
        pass

    return None


# (4) 正規化ユーティリティ_列名/コード/区分
def _norm_col(col: str) -> str:
    """
    CSV列名の正規化：
    - 前後空白除去
    - 全角スペース→半角
    """
    if col is None:
        return ""
    return str(col).strip().replace("\u3000", " ")


def _norm_code(code: str) -> str:
    """
    銘柄コード正規化：
    - 前後空白除去
    - 全角英数字 → 半角
    """
    if not code:
        return ""
    s = str(code).strip()
    # 全角英数字を半角に
    s = s.translate(str.maketrans(
        "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺ",
        "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ))
    return s


def to_category(market: str) -> str:
    """
    市場・商品区分 → category
    """
    m = (market or "").strip()

    if "ETF" in m or "ETN" in m:
        return "ETF"
    if "REIT" in m:
        return "REIT"
    if "PRO" in m:
        return "PRO"
    if "内国株式" in m or "プライム" in m or "スタンダード" in m or "グロース" in m:
        return "株"
    return "その他"


def initial_status_from_category(category: str) -> str:
    """
    category に応じた初期 status
    """
    if category == "株":
        return "pending"
    return "skip"


# (5) IDmap（全列保持）入出力_関数群（必要列のみ更新して他列は保持）
# (5-1) 既存IDmap読込_全列保持（BOM対応）
def load_existing_idmap(path: str) -> Dict[str, Dict[str, str]]:
    """
    既存IDmapを全列保持で読み込む。
    - code をキーに行辞書を保持
    - CSVに追加列があっても保持（破棄しない）
    """
    data: Dict[str, Dict[str, str]] = {}
    if not os.path.exists(path):
        return data

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            code = (row.get("code") or "").strip()
            if not code:
                continue
            # ★全列保持
            data[code] = {k: (v if v is not None else "") for k, v in row.items()}
            data[code]["code"] = code  # 念のため正規化
    return data

# (5-2) IDmap保存_全列保持（原子的保存）
def save_idmap(rows_map: Dict[str, Dict[str, str]], out_csv: str = IDMAP_CSV) -> str:
    """
    IDmapを全列保持で保存する。
    - 既存の追加列を消さない
    - 新規に追加された列も出力対象に含める
    """
    # 1) すべての列名を収集
    all_keys: set = set()
    for r in rows_map.values():
        all_keys.update(r.keys())

    # 2) 列の並び順（基本列を先頭、残りは既存列として末尾）
    base_cols = ["code", "ID", "URL", "name", "UPDATE", "category", "status", "industry_33"]
    extra_cols = [c for c in all_keys if c not in base_cols]
    cols = base_cols + sorted(extra_cols)

    # 3) ソート
    def _sort_key(r: Dict[str, str]):
        c = str(r.get("code", ""))
        m = re.match(r"^\d+", c)
        return (int(m.group(0)) if m else 10**12, c)

    sorted_rows = sorted(rows_map.values(), key=_sort_key)

    # 4) 原子的保存
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix="idmap_", suffix=".csv", dir=os.path.dirname(out_csv))
    os.close(fd)

    try:
        with open(tmp, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=cols)
            writer.writeheader()
            for r in sorted_rows:
                out_row = {}
                for c in cols:
                    v = r.get(c)
                    out_row[c] = "" if v is None else str(v)
                writer.writerow(out_row)
        os.replace(tmp, out_csv)
    finally:
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass

    return out_csv


# (6) 入力CSV→（code, category, industry_33）抽出_関数群
# (6-1) 入力CSVから対象（code, category, market_raw, industry_33）一覧を作る
def load_items_from_listing_csv(listing_csv: str) -> List[Dict[str, str]]:
    if not os.path.exists(listing_csv):
        raise FileNotFoundError(f"入力CSVが見つかりません: {listing_csv}")

    try:
        df = pd.read_csv(listing_csv, dtype=str, encoding="utf-8-sig").fillna("")
    except Exception:
        df = pd.read_csv(listing_csv, dtype=str, encoding="cp932").fillna("")

    df.columns = [_norm_col(c) for c in df.columns]

    if "コード" not in df.columns:
        raise ValueError(f"列 'コード' が見つかりません。実列={list(df.columns)}")
    if "市場・商品区分" not in df.columns:
        df["市場・商品区分"] = ""

    # ★ 33業種区分（無い場合は空欄でOK）
    if "33業種区分" not in df.columns:
        df["33業種区分"] = ""

    items: List[Dict[str, str]] = []
    for _, r in df.iterrows():
        code = _norm_code(r.get("コード", ""))
        if not code:
            continue
        market = str(r.get("市場・商品区分", "")).strip()
        cat = to_category(market)

        ind33 = str(r.get("33業種区分", "")).strip()

        items.append({
            "code": code,
            "category": cat,
            "market_raw": market,
            "industry_33": ind33,
        })

    # 重複（同一コード）を除去：最初の出現を優先
    uniq: Dict[str, Dict[str, str]] = {}
    for it in items:
        if it["code"] not in uniq:
            uniq[it["code"]] = it

    def _sort_code(c: str):
        m = re.match(r"^\d+", c)
        return (int(m.group(0)) if m else 10**12, c)

    return sorted(list(uniq.values()), key=lambda it: _sort_code(it["code"]))


# (7) 進捗表示_1銘柄1行（最も確実で分かりやすい）
def print_progress_line(i: int,
                        total: int,
                        code: str,
                        category: str,
                        status: str,
                        counts: Dict[str, int]) -> None:
    """
    1銘柄ごとに進捗を1行で表示する。

    表示例：
    [12/4009] 130A  株   status=ok         ok=10 skip=0 not_found=1 err=0
    """
    print(
        f"[{i}/{total}] {code:<6} "
        f"{category:<4} "
        f"status={status:<9} "
        f"ok={counts.get('ok',0)} "
        f"skip={counts.get('skip',0)} "
        f"not_found={counts.get('not_found',0)} "
        f"pending={counts.get('pending',0)} "
        f"err={counts.get('err',0)}"
    )


# (8) メイン処理_IDmapに category/status/industry_33 を持たせて更新（他列/他行は保持）
def build_idmap_with_category_status(listing_csv: str = LISTING_CSV,
                                     out_csv: str = IDMAP_CSV,
                                     sess: Optional[requests.Session] = None,
                                     sleep_sec: float = SLEEP_BASE) -> pd.DataFrame:
    sess = sess or make_session()
    rows_map = load_existing_idmap(out_csv)

    items = load_items_from_listing_csv(listing_csv)
    today_jst = datetime.now(TZ).strftime("%Y-%m-%d")

    counts = {"ok": 0, "skip": 0, "not_found": 0, "pending": 0, "err": 0}
    total = len(items)

    for i, it in enumerate(items, 1):
        code = it["code"]
        category = it["category"]
        industry_33 = it.get("industry_33", "")

        # 既存レコード（なければ新規枠：ただし他列も将来追加しやすいよう dict で用意）
        rec = rows_map.get(code, {
            "code": code, "ID": "", "URL": "", "name": "", "UPDATE": "",
            "category": "", "status": "", "industry_33": ""
        })

        # 入力CSVを正として更新（市場区分/業種は毎回追随）
        rec["category"] = category
        rec["industry_33"] = industry_33

        # 既存で OK（IDあり）なら “更新しない” を維持して即スキップ
        # ※category/industry_33 の追随はすでに反映済み
        if rec.get("ID") and rec.get("status") == "ok":
            counts["ok"] += 1
            rows_map[code] = rec
            print_progress_line(i, total, code, category, rec["status"], counts)
            continue

        # 初期status（未設定 or 空の場合）
        if not rec.get("status"):
            rec["status"] = initial_status_from_category(category)

        # category が株以外は、現段階では EID 取得をしない（＝skip）
        if category != "株":
            rec["status"] = "skip"
            rec["UPDATE"] = today_jst
            rows_map[code] = rec
            counts["skip"] += 1
            print_progress_line(i, total, code, category, rec["status"], counts)
            continue

        # ここから「株」：EID を取得する（未取得 or ok以外）
        rec["status"] = "pending"
        rows_map[code] = rec
        counts["pending"] += 1
        print_progress_line(i, total, code, category, rec["status"], counts)

        try:
            eid = resolve_eid_from_code(code, sess, wait=sleep_sec)
            if not eid:
                rec["ID"] = ""
                rec["URL"] = ""
                rec["status"] = "not_found"
                rec["UPDATE"] = today_jst
                rows_map[code] = rec

                counts["not_found"] += 1
                counts["pending"] = max(0, counts["pending"] - 1)

                print_progress_line(i, total, code, category, rec["status"], counts)

                polite_sleep(sleep_sec)
                save_idmap(rows_map, out_csv)
                continue

            results_url = RESULTS_URL.format(eid=eid)

            # name 取得（results優先→codeページ）
            name = ""
            try:
                polite_sleep(sleep_sec)
                html_results = fetch_html(results_url, sess)
                name = extract_name_from_html(html_results) or ""
            except Exception:
                try:
                    polite_sleep(sleep_sec)
                    html_code = fetch_html(BASE_BY_CODE.format(code=code), sess)
                    name = extract_name_from_html(html_code) or ""
                except Exception:
                    name = ""

            rec["ID"] = eid
            rec["URL"] = results_url
            rec["name"] = name
            rec["UPDATE"] = today_jst
            rec["status"] = "ok"
            rows_map[code] = rec

            counts["ok"] += 1
            counts["pending"] = max(0, counts["pending"] - 1)

            print_progress_line(i, total, code, category, rec["status"], counts)

            save_idmap(rows_map, out_csv)

        except Exception:
            rec["status"] = "not_found"
            rec["UPDATE"] = today_jst
            rows_map[code] = rec

            counts["err"] += 1
            counts["pending"] = max(0, counts["pending"] - 1)

            print_progress_line(i, total, code, category, rec["status"], counts)

        finally:
            polite_sleep(sleep_sec)

    # ★ listing_csv に無い既存行も rows_map に残っているため、ここで消えない
    df = pd.DataFrame(list(rows_map.values()))
    return df


# (9) 実行例
if __name__ == "__main__":
    session = make_session()

    df_idmap = build_idmap_with_category_status(
        listing_csv=LISTING_CSV,
        out_csv=IDMAP_CSV,
        sess=session,
        sleep_sec=SLEEP_BASE,
    )

    # 結果確認（先頭10件）
    try:
        print("\n[preview] Data/01_IDmap.csv (head)")
        print(df_idmap.head(10).to_string(index=False))
    except Exception:
        pass

[1/4425] 130A   株    status=ok        ok=1 skip=0 not_found=0 pending=0 err=0
[2/4425] 131A   PRO  status=skip      ok=1 skip=1 not_found=0 pending=0 err=0
[3/4425] 132A   PRO  status=skip      ok=1 skip=2 not_found=0 pending=0 err=0
[4/4425] 133A   ETF  status=skip      ok=1 skip=3 not_found=0 pending=0 err=0
[5/4425] 134A   PRO  status=skip      ok=1 skip=4 not_found=0 pending=0 err=0
[6/4425] 135A   株    status=ok        ok=2 skip=4 not_found=0 pending=0 err=0
[7/4425] 136A   PRO  status=skip      ok=2 skip=5 not_found=0 pending=0 err=0
[8/4425] 137A   株    status=ok        ok=3 skip=5 not_found=0 pending=0 err=0
[9/4425] 138A   株    status=ok        ok=4 skip=5 not_found=0 pending=0 err=0
[10/4425] 139A   PRO  status=skip      ok=4 skip=6 not_found=0 pending=0 err=0
[11/4425] 140A   ETF  status=skip      ok=4 skip=7 not_found=0 pending=0 err=0
[12/4425] 141A   株    status=ok        ok=5 skip=7 not_found=0 pending=0 err=0
[13/4425] 142A   株    status=ok        ok=6 skip=7 not_found=